# Rats: A Normal Hierarchical Model

Stan, run with [CmdStanPy](https://mc-stan.org/cmdstanpy/).

The first cell installs CmdStan, which compiles a toolchain and takes several minutes on a fresh Colab runtime.

In [ ]:
%pip install -q cmdstanpy

import cmdstanpy
cmdstanpy.install_cmdstan(progress=True)

## The model

In [ ]:
model_code = '''
data {
  int<lower=1> N;
  int<lower=1> T;
  array[T] real x;
  real xbar;
  array[N, T] real Y;
}

parameters {
  real<lower=0> tau_c;
  real alpha_c;
  real<lower=0> alpha_tau;
  real beta_c;
  real<lower=0> beta_tau;
  array[N] real alpha;
  array[N] real beta;
}

transformed parameters {
  array[N, T] real mu;
  for (i in 1:N) {
    for (j in 1:T) {
      mu[i,j] = alpha[i] + beta[i] * (x[j] - xbar);
    }
  }
}

model {
  alpha ~ normal(alpha_c, 1.0 / sqrt(alpha_tau));
  beta ~ normal(beta_c, 1.0 / sqrt(beta_tau));
  for (i in 1:N) {
    for (j in 1:T) {
      Y[i,j] ~ normal(mu[i,j], 1.0 / sqrt(tau_c));
    }
  }
  tau_c ~ gamma(0.001, 0.001);
  alpha_c ~ normal(0.0, 1.0 / sqrt(1.0E-6));
  alpha_tau ~ gamma(0.001, 0.001);
  beta_c ~ normal(0.0, 1.0 / sqrt(1.0E-6));
  beta_tau ~ gamma(0.001, 0.001);
}

generated quantities {
  real sigma;
  real alpha0;
  sigma = 1 / sqrt(tau_c);
  alpha0 = alpha_c - xbar * beta_c;
}

'''

with open("model.stan", "w") as f:
    f.write(model_code)

print(model_code)

## Data and initial values

In [ ]:
import json

data = json.loads(r'''
{
  "N": 30,
  "T": 5,
  "x": [
    8,
    15,
    22,
    29,
    36
  ],
  "xbar": 22,
  "Y": [
    [
      151,
      199,
      246,
      283,
      320
    ],
    [
      145,
      199,
      249,
      293,
      354
    ],
    [
      147,
      214,
      263,
      312,
      328
    ],
    [
      155,
      200,
      237,
      272,
      297
    ],
    [
      135,
      188,
      230,
      280,
      323
    ],
    [
      159,
      210,
      252,
      298,
      331
    ],
    [
      141,
      189,
      231,
      275,
      305
    ],
    [
      159,
      201,
      248,
      297,
      338
    ],
    [
      177,
      236,
      285,
      350,
      376
    ],
    [
      134,
      182,
      220,
      260,
      296
    ],
    [
      160,
      208,
      261,
      313,
      352
    ],
    [
      143,
      188,
      220,
      273,
      314
    ],
    [
      154,
      200,
      244,
      289,
      325
    ],
    [
      171,
      221,
      270,
      326,
      358
    ],
    [
      163,
      216,
      242,
      281,
      312
    ],
    [
      160,
      207,
      248,
      288,
      324
    ],
    [
      142,
      187,
      234,
      280,
      316
    ],
    [
      156,
      203,
      243,
      283,
      317
    ],
    [
      157,
      212,
      259,
      307,
      336
    ],
    [
      152,
      203,
      246,
      286,
      321
    ],
    [
      154,
      205,
      253,
      298,
      334
    ],
    [
      139,
      190,
      225,
      267,
      302
    ],
    [
      146,
      191,
      229,
      272,
      302
    ],
    [
      157,
      211,
      250,
      285,
      323
    ],
    [
      132,
      185,
      237,
      286,
      331
    ],
    [
      160,
      207,
      257,
      303,
      345
    ],
    [
      169,
      216,
      261,
      295,
      333
    ],
    [
      157,
      205,
      248,
      289,
      316
    ],
    [
      137,
      180,
      219,
      258,
      291
    ],
    [
      153,
      200,
      244,
      286,
      324
    ]
  ]
}
''')
inits = json.loads(r'''
{
  "alpha": [
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250,
    250
  ],
  "beta": [
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6,
    6
  ],
  "alpha_c": 150,
  "beta_c": 10,
  "tau_c": 1,
  "alpha_tau": 1,
  "beta_tau": 1
}
''')

with open("data.json", "w") as f:
    json.dump(data, f)
with open("inits.json", "w") as f:
    json.dump(inits, f)

data

## Sample

In [ ]:
model = cmdstanpy.CmdStanModel(stan_file="model.stan")
fit = model.sample(
    data="data.json",
    inits="inits.json",
    chains=1,
    iter_warmup=1000,
    iter_sampling=1000,
    seed=42,
)

## Results

In [ ]:
print(fit.summary())
print(fit.diagnose())